# Event-Level Analysis and Visualization (CORE)

**Chapter 3: Exploratory Data Analysis in Soccer**

## Learning Objectives

By the end of this notebook, you will be able to:
- Load and explore event-level data from StatsBomb
- Filter events by type, team, and player
- Group and aggregate events to create meaningful summaries
- Calculate pass accuracy, shot statistics, and expected goals (xG)
- Create effective visualizations for event data
- Understand the relationship between event data and match outcomes

## Prerequisites

- Completed previous notebooks (01-03)
- Understanding of pandas filtering and grouping

## Setup and Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Loading Event Data

Event data is stored in separate JSON files, one per match. Each file contains every on-ball action (pass, shot, tackle, etc.) that occurred during that match.

Let's start by loading events from a single match to understand the structure.

In [ ]:
# Define base path
BASE = Path("open-data/data")

# Load events from a single match (example match ID)
match_id = 22921  # France vs Korea Republic
events = json.load(open(BASE / f"events/{match_id}.json", "r"))

# Convert to DataFrame
events_df = pd.json_normalize(events, sep="_")
events_df["match_id"] = match_id

print(f"Loaded {len(events_df)} events from match {match_id}")
events_df.head()

## Understanding Event Types

The first step in event-level analysis is understanding what types of events we have. Each event has a `type_name` that describes the action.

In [ ]:
# Count events by type
event_counts = events_df["type_name"].value_counts()
print("\nTop 10 Event Types:")
print(event_counts.head(10))

# Visualize
plt.figure(figsize=(10, 6))
sns.barplot(y=event_counts.head(10).index, x=event_counts.head(10).values, orient='h')
plt.xlabel("Count")
plt.title("Top 10 Event Types in Match")
plt.tight_layout()
plt.show()

**What this tells us:**
- Passes dominate the event log (typically 60-70% of all events)
- Ball receipts are also very common (one for each successful pass)
- Duels, carries, and pressure events follow
- Shots and goals are much rarer but more decisive

This distribution is exactly what we'd expect in a typical soccer match.

## Filtering: Focusing on Specific Events

The simplest form of analysis is filtering—selecting only the events we care about. Let's look at passes by a specific team.

In [ ]:
# Filter for passes by France
france_passes = events_df.query("type_name == 'Pass' and team_name == 'France Women\'s'")
print(f"France made {len(france_passes)} passes in this match")

# Look at the first few passes
france_passes[["minute", "second", "player_name", "type_name", "pass_outcome_name"]].head(10)

## Understanding Pass Outcomes

In StatsBomb data, a completed pass has a **missing** (NaN) value in `pass_outcome_name`. Any other value ("Incomplete", "Out", "Pass Offside", etc.) indicates the pass failed.

In [ ]:
# Check pass outcomes
print("\nPass Outcomes:")
print(events_df[events_df["type_name"] == "Pass"]["pass_outcome_name"].value_counts(dropna=False))

# Create a 'completed' boolean column
passes = events_df.query("type_name == 'Pass'").copy()
passes["completed"] = passes["pass_outcome_name"].isna()

print(f"\nTotal passes: {len(passes)}")
print(f"Completed: {passes['completed'].sum()}")
print(f"Incomplete: {(~passes['completed']).sum()}")
print(f"Pass accuracy: {passes['completed'].mean():.1%}")

## Grouping: Per-Match Pass Summaries

Now let's load multiple matches and create per-match summaries for each team. This is where pandas' `groupby` becomes powerful.

In [ ]:
# Load matches to get match IDs
matches = json.load(open(BASE / "matches/72/30.json", "r"))
matches_df = pd.json_normalize(matches)
match_ids = matches_df["match_id"].tolist()[:5]  # Start with first 5 matches

# Load events from multiple matches
all_events = []
for mid in match_ids:
    events = json.load(open(BASE / f"events/{mid}.json", "r"))
    events_df_temp = pd.json_normalize(events, sep="_")
    events_df_temp["match_id"] = mid
    all_events.append(events_df_temp)

# Combine into single DataFrame
events_df = pd.concat(all_events, ignore_index=True)
print(f"Loaded {len(events_df)} events from {len(match_ids)} matches")

In [ ]:
# Calculate per-match passing statistics
passes = events_df.query("type_name == 'Pass'").copy()
passes["completed"] = passes["pass_outcome_name"].isna()

per_match_pass = (
    passes.groupby(["match_id", "team_name"])
    .agg(
        attempted=("type_name", "count"),
        completed=("completed", "sum")
    )
    .reset_index()
)

per_match_pass["pass_accuracy"] = (
    per_match_pass["completed"] / per_match_pass["attempted"]
)

print("\nPer-Match Pass Statistics:")
per_match_pass.head(10)

## Visualizing Pass Accuracy by Team

Let's create a bar chart showing average pass accuracy for each team across all their matches.

In [ ]:
# Calculate average pass accuracy per team
team_pass_accuracy = (
    per_match_pass.groupby("team_name")["pass_accuracy"]
    .mean()
    .sort_values(ascending=False)
)

# Create horizontal bar chart
plt.figure(figsize=(10, 8))
sns.barplot(x=team_pass_accuracy.values, y=team_pass_accuracy.index, orient='h')
plt.xlabel("Pass Accuracy")
plt.title("Average Pass Accuracy by Team")
plt.xlim(0.5, 1.0)
plt.tight_layout()
plt.show()

## Analyzing Shots and Goals

Passing tells us who controls the ball, but goals decide matches. Let's analyze shot data and expected goals (xG).

In [ ]:
# Filter for shots
shots = events_df.query("type_name == 'Shot'").copy()

# Create goal indicator
shots["is_goal"] = shots["shot_outcome_name"].eq("Goal")

print(f"Total shots: {len(shots)}")
print(f"Goals: {shots['is_goal'].sum()}")
print(f"Conversion rate: {shots['is_goal'].mean():.1%}")

# Check if xG data is available
xg_col = "shot_statsbomb_xg" if "shot_statsbomb_xg" in shots.columns else None
if xg_col:
    print(f"\nAverage xG per shot: {shots[xg_col].mean():.3f}")
    print(f"Total xG: {shots[xg_col].sum():.2f}")

## Per-Match Shot Statistics

Let's create a summary table of shots, goals, and xG for each team in each match.

In [ ]:
# Build aggregation dictionary
agg_dict = {
    "shots": ("type_name", "count"),
    "goals": ("is_goal", "sum")
}

if xg_col:
    agg_dict["xg"] = (xg_col, "sum")

# Create per-match shot summary
per_match_shot = (
    shots.groupby(["match_id", "team_name"])
    .agg(**agg_dict)
    .reset_index()
)

print("\nPer-Match Shot Statistics:")
per_match_shot.head(10)

## Team-Level Aggregation

Roll up to tournament-level summaries for each team.

In [ ]:
# Aggregate to team level
agg_ops = {"shots": "sum", "goals": "sum"}
if "xg" in per_match_shot.columns:
    agg_ops["xg"] = "sum"

team_summary = per_match_shot.groupby("team_name").agg(agg_ops)

# Calculate xG per shot (shot quality metric)
if "xg" in team_summary.columns:
    team_summary["xg_per_shot"] = (
        team_summary["xg"] / team_summary["shots"].clip(lower=1)
    )

# Sort by goals
team_summary = team_summary.sort_values("goals", ascending=False)

print("\nTeam Summary (Shots and Goals):")
print(team_summary)

## Visualizing Shot Volume vs Goals

Let's create a scatter plot to see the relationship between shot volume and goals scored.

In [ ]:
# Calculate per-match averages
team_averages = (
    per_match_shot.groupby("team_name")
    .agg(
        shots_per_match=("shots", "mean"),
        goals_per_match=("goals", "mean")
    )
    .reset_index()
)

# Create scatter plot
plt.figure(figsize=(10, 7))
sns.scatterplot(data=team_averages, x="shots_per_match", y="goals_per_match", s=100)

# Add trend line
x = team_averages["shots_per_match"]
y = team_averages["goals_per_match"]
m, b = np.polyfit(x, y, 1)
plt.plot(x, m*x + b, color='blue', linestyle='--', alpha=0.8, label='Trend')

plt.xlabel("Shots per Match")
plt.ylabel("Goals per Match")
plt.title("Shot Volume vs Goals Scored")
plt.legend()
plt.tight_layout()
plt.show()

**What this tells us:**
- Clear positive correlation: more shots → more goals
- Teams above the trend line are more clinical (better conversion)
- Teams below the line need to improve shot quality or finishing
- The scatter shows that both volume AND quality matter

## Distribution of Shot Quality (xG)

Let's visualize the distribution of shot quality using expected goals.

In [ ]:
if xg_col:
    plt.figure(figsize=(10, 6))
    sns.histplot(data=shots, x=xg_col, bins=25, color="salmon")
    plt.xlabel("Shot xG (Expected Goals)")
    plt.ylabel("Count")
    plt.title("Distribution of Shot Quality (xG)")
    plt.tight_layout()
    plt.show()
else:
    print("xG data not available in this dataset")

**What this tells us:**
- Most shots have low xG (< 0.1), meaning low probability of scoring
- High-quality chances (xG > 0.3) are rare and precious
- This distribution is normal for soccer—most shots are from difficult positions
- Teams that can create more high-xG chances will be more successful

## Summary

In this notebook, we learned how to:

1. **Load event-level data** from StatsBomb JSON files
2. **Filter events** by type, team, and player
3. **Group and aggregate** events to create meaningful summaries
4. **Calculate key metrics**:
   - Pass accuracy (completed / attempted)
   - Shot conversion rate (goals / shots)
   - Expected goals (xG) totals and averages
5. **Create visualizations** to reveal patterns and insights

These are the fundamental building blocks of soccer analytics. From here, we can create more sophisticated analyses, build predictive models, and develop tactical insights.

## Next Steps

In the extras notebooks, we'll explore:
- Advanced visualizations using mplsoccer (shot maps, heatmaps)
- Passing network analysis
- Complete case study of Japan Women's at WWC 2019

## Practice Exercises

1. **Pass Length Analysis**: Calculate the average pass length for each team (use `pass_length` column if available)
2. **Shot Distance**: Analyze the distribution of shot distances and compare teams
3. **Player-Level Analysis**: Find the top 10 players by number of passes completed
4. **Time-Based Analysis**: Analyze how shot frequency changes throughout the match (by minute)
5. **Defensive Actions**: Count tackles, interceptions, and clearances by team